# A-Network — 物理神经网络训练

基于 PyTorch 的 3D 张量场物理神经网络。
Token 注入 80×80×80 场中，经 20 步 26-邻域扩散传播后读出为 logits。

**仓库**: https://github.com/Mn3TR/a-network

## 1. 环境准备

克隆仓库并安装依赖。

In [ ]:
# 克隆仓库
!git clone https://github.com/Mn3TR/a-network.git
%cd a-network

In [ ]:
# 安装依赖
!pip install -r requirements.txt

In [ ]:
# 检查 GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 挂载 Google Drive（可选）

挂载 Drive 后，训练好的权重和日志会持久保存，即使 Colab 断开也不会丢失。
如果不挂载，权重仅保存在虚拟机临时目录中。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 在 Drive 中创建输出目录
!mkdir -p /content/drive/MyDrive/a-network/output
!mkdir -p /content/drive/MyDrive/a-network/log
print('Drive 已挂载，输出将保存到 /content/drive/MyDrive/a-network/')

## 3. 配置训练参数

可在此修改超参数。默认值与 C++ 版一致。

In [ ]:
import sys
sys.path.insert(0, '.')

from src.anetwork.config import ANetworkConfig, TrainConfig
from src.anetwork.model import ANetwork
from src.anetwork.tokenizer import TokenizerWrapper
from src.anetwork.data import load_data

# ========== 训练参数 ==========
TRAIN_EPOCHS = 10          # 训练轮数（Colab 上建议先用少量 epoch 测试）
GRAD_ACCUM = 4             # 梯度积累步数
LEARNING_RATE = 1e-4

# 是否使用 Drive 保存输出
USE_DRIVE = False  # 挂载了 Drive 的话改为 True

# ========== 网络参数（通常不改） ==========
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# 加载 tokenizer
tokenizer = TokenizerWrapper('tokenizer/tokenizer.json')
print(f'Vocab size: {tokenizer.vocab_size}')

# 加载数据
tokens = load_data('dataset/', tokenizer)
print(f'Training tokens: {len(tokens)}')

# 创建网络
a_cfg = ANetworkConfig(vocab_size=tokenizer.vocab_size)
net = ANetwork(a_cfg, device=DEVICE)
net.to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in net.parameters()):,}')

# 优化器
optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)

## 4. 开始训练

每个 epoch 结束后自动保存权重到 output/。
场快照保存在 log/ 目录，可用后面小节的可视化工具查看。

In [ ]:
import math
import time
from pathlib import Path

total_steps = len(tokens) - 1
output_dir = '/content/drive/MyDrive/a-network' if USE_DRIVE else '.'

for epoch in range(TRAIN_EPOCHS):
    # 余弦学习率退火
    if TRAIN_EPOCHS > 1:
        progress = epoch / (TRAIN_EPOCHS - 1)
        factor = (1.0 + math.cos(math.pi * progress)) * 0.5
        lr = 1e-6 + (LEARNING_RATE - 1e-6) * factor
        for pg in optimizer.param_groups:
            pg['lr'] = lr

    net.reset_state()
    net.train()
    epoch_loss = 0.0
    epoch_start = time.time()
    optimizer.zero_grad()

    for step_idx in range(total_steps):
        input_id = torch.tensor(tokens[step_idx], device=DEVICE)
        target_id = torch.tensor(tokens[step_idx + 1], device=DEVICE)

        loss = net.train_step(input_id, target_id)
        loss.backward()
        epoch_loss += loss.item()

        if (step_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        # 每 10 步打印一次
        if (step_idx + 1) % max(1, total_steps // 10) == 0:
            print(f'  epoch {epoch} [{step_idx+1}/{total_steps}] loss={loss.item():.4f}')

    if total_steps % GRAD_ACCUM != 0:
        optimizer.step()
        optimizer.zero_grad()

    epoch_s = time.time() - epoch_start
    avg_loss = epoch_loss / total_steps
    print(f'Epoch {epoch}: avg_loss={avg_loss:.4f}  time={epoch_s:.1f}s  lr={lr:.8f}')

    # 保存快照
    net.save(f'{output_dir}/output/weights_epoch{epoch}.pt')

print('Training done!')

## 5. 生成文本

用训练好的模型自回归生成文本。

In [ ]:
net.eval()
seed_text = 'Time'
seed_ids = tokenizer.encode(seed_text)[:3]
print(f'Seed: "{seed_text}"')

generated = net.generate(seed_ids, max_tokens=50)
output = tokenizer.decode(generated)
print(f'Output: "{output}"')

## 6. 场状态可视化

查看训练过程中场状态的演化。需要先安装 matplotlib（已包含在 requirements.txt）。

In [ ]:
# 在训练过程中或之后，查看场状态
import numpy as np
import matplotlib.pyplot as plt

field = net.field.detach().cpu().numpy().reshape(80, 80, 80)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for i, z in enumerate([20, 40, 60]):
    im = axes[i].imshow(field[:, :, z], cmap='viridis')
    axes[i].set_title(f'z={z}')
    axes[i].axis('off')
fig.colorbar(im, ax=axes, shrink=0.6)
plt.show()